In [ ]:
!pip install scikit-learn joblib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping
import joblib

In [ ]:
file_name = "/content/anomly_detection.csv"
df = pd.read_csv(file_name)

In [ ]:
df.head()

,sensor_timestamp,humidity,temperature,coldroom_name
0,06:59 PM - 25/02/2026,64.1,24.4,Cold Room 1
1,06:58 PM - 25/02/2026,64.0,24.4,Cold Room 1
2,06:57 PM - 25/02/2026,64.2,24.3,Cold Room 1
3,06:56 PM - 25/02/2026,64.1,24.3,Cold Room 1
4,06:55 PM - 25/02/2026,64.1,24.3,Cold Room 1


In [ ]:

def normalize_name(name):
    return name.replace(" ", "").strip().lower()

df["coldroom_name"] = df["coldroom_name"].apply(normalize_name)


In [ ]:
df.head()

,sensor_timestamp,humidity,temperature,coldroom_name,temp_diff,rolling_mean,rolling_std
0,2026-02-19 00:00:00,64.9,9.8,coldroom1,0.0,0.0,0.0
1,2026-02-19 00:01:00,64.6,10.6,coldroom1,0.8,0.0,0.0
2,2026-02-19 00:02:00,64.8,11.2,coldroom1,0.6,0.0,0.0
3,2026-02-19 00:03:00,64.6,11.7,coldroom1,0.5,0.0,0.0
4,2026-02-19 00:04:00,64.5,12.3,coldroom1,0.6,0.0,0.0


In [ ]:
df["sensor_timestamp"] = pd.to_datetime(
    df["sensor_timestamp"],
    format="%I:%M %p - %d/%m/%Y"
)
df = df.sort_values(["coldroom_name","sensor_timestamp"]).reset_index(drop=True)

In [ ]:
df["temp_diff"] = df.groupby("coldroom_name")["temperature"].diff()
df["rolling_mean"] = df.groupby("coldroom_name")["temperature"].rolling(12).mean().reset_index(0,drop=True)
df["rolling_std"] = df.groupby("coldroom_name")["temperature"].rolling(12).std().reset_index(0,drop=True)


In [ ]:
df.head()

,sensor_timestamp,humidity,temperature,coldroom_name,temp_diff,rolling_mean,rolling_std
0,2026-02-19 00:00:00,64.9,9.8,coldroom1,NaN,NaN,NaN
1,2026-02-19 00:01:00,64.6,10.6,coldroom1,0.8,NaN,NaN
2,2026-02-19 00:02:00,64.8,11.2,coldroom1,0.6,NaN,NaN
3,2026-02-19 00:03:00,64.6,11.7,coldroom1,0.5,NaN,NaN
4,2026-02-19 00:04:00,64.5,12.3,coldroom1,0.6,NaN,NaN


In [ ]:
df = df.fillna(0)

TIME_STEPS = 30


In [ ]:
def create_sequences(data, time_steps=30):
    sequences = []
    for i in range(len(data) - time_steps):
        sequences.append(data[i:(i + time_steps)])
    return np.array(sequences)

In [ ]:
def build_model(n_features):
    input_layer = Input(shape=(TIME_STEPS, n_features))
    encoded = LSTM(64, activation="relu", return_sequences=False)(input_layer)
    encoded = RepeatVector(TIME_STEPS)(encoded)
    decoded = LSTM(64, activation="relu", return_sequences=True)(encoded)
    decoded = TimeDistributed(Dense(n_features))(decoded)

    model = Model(inputs=input_layer, outputs=decoded)
    model.compile(optimizer="adam", loss="mse")
    return model

In [ ]:
room_scalers = {}
room_thresholds = {}
all_sequences = []

In [ ]:
for room in df["coldroom_name"].unique():

    room_df = df[df["coldroom_name"] == room].copy()

    features = room_df[[
        "temperature",
        "humidity",
        "temp_diff",
        "rolling_mean",
        "rolling_std"
    ]]

    scaler = StandardScaler()
    scaled = scaler.fit_transform(features)

    room_scalers[room] = scaler

    X_room = create_sequences(scaled, TIME_STEPS)

    all_sequences.append(X_room)


In [ ]:
X = np.vstack(all_sequences)

model = build_model(X.shape[2])

early_stop = EarlyStopping(monitor="loss", patience=5, restore_best_weights=True)

model.fit(
    X, X,
    epochs=40,
    batch_size=32,
    shuffle=False,
    callbacks=[early_stop]
)


Epoch 1/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - loss: 0.7360
Epoch 2/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.4105
Epoch 3/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.3680
Epoch 4/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.3370
Epoch 5/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.3284
Epoch 6/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 0.3242
Epoch 7/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.3024
Epoch 8/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.2895
Epoch 9/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.2975
Epoch 10/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.2807
Epoch 11/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 0.2757
Epoch 12/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.2676
Epoch 13/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.2762
Epoch 14/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.2544
Epoch 15/40
680/680 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - 

In [ ]:
for room in df["coldroom_name"].unique():

    room_df = df[df["coldroom_name"] == room].copy()

    features = room_df[[
        "temperature",
        "humidity",
        "temp_diff",
        "rolling_mean",
        "rolling_std"
    ]]

    scaler = room_scalers[room]
    scaled = scaler.transform(features)

    X_room = create_sequences(scaled, TIME_STEPS)

    if len(X_room) < 10:
        continue

    X_pred = model.predict(X_room, verbose=0)

    reconstruction_error = np.mean(np.power(X_room - X_pred, 2), axis=(1,2))
    threshold = np.percentile(reconstruction_error, 95)

    room_thresholds[room] = threshold

In [ ]:
model.save("lstm_autoencoder_model.h5")
joblib.dump(room_scalers, "lstm_scaler.pkl")
joblib.dump(room_thresholds, "lstm_threshold.pkl")

['lstm_threshold.pkl']

In [ ]:
print("✔ Saved:")
print("lstm_autoencoder_model.h5")
print("lstm_scaler.pkl (contains per-room scalers)")
print("lstm_threshold.pkl (contains per-room thresholds)")

✔ Saved:
lstm_autoencoder_model.h5
lstm_scaler.pkl (contains per-room scalers)
lstm_threshold.pkl (contains per-room thresholds)
